In [1]:
import re
import collections
import pprint

import numpy as np

GLOBAL_RANDOM_SEED = 42

### **Задание 1. Корпус и токенизация.**
- Скачать текст.
- Привести к нижнему регистру, разбить на слова регулярным выражением, знаки препинания либо выбросить, либо сделать отдельными токенами (выбор объяснить).
- Статистика: число слов, размер словаря, двадцать самых частых слов.
- Отложить последние 10% текста для проверки, модель их не видит.

In [2]:
def tokenize_text(text):
    word_pattern = r"[а-яё]+(?:-[а-яё]+)*"
    return re.findall(word_pattern, re.sub("ё", "е", text.lower()))


def load_oblomov_tokens():
    with open("oblomov.txt", "r", encoding="utf8") as file:
        return tokenize_text(file.read())


def run_exercise_1():
    np.random.seed(GLOBAL_RANDOM_SEED)

    tokens = load_oblomov_tokens()

    token_counter = collections.Counter(tokens)
    tokens_length = len(tokens)
    train_size = int(tokens_length * 0.9)
    train_tokens = tokens[:train_size]
    test_tokens = tokens[train_size:]

    print(
        f"Общее количество слов - {tokens_length}",
        f"Размер обучающей выборки - {len(train_tokens)}",
        f"Размер тестовой выборки - {len(test_tokens)}",
        f"Размер словаря - {len(token_counter)}",
        "Наиболее часто встречающиеся слова:",
        sep="\n",
    )
    pprint.pprint(token_counter.most_common(20))


run_exercise_1()

Общее количество слов - 147313
Размер обучающей выборки - 132581
Размер тестовой выборки - 14732
Размер словаря - 25471
Наиболее часто встречающиеся слова:
[('и', 6015),
 ('не', 3896),
 ('в', 3524),
 ('он', 2626),
 ('что', 2407),
 ('на', 2400),
 ('с', 2038),
 ('она', 1634),
 ('а', 1574),
 ('как', 1534),
 ('я', 1443),
 ('все', 1012),
 ('обломов', 909),
 ('это', 903),
 ('его', 903),
 ('у', 879),
 ('ты', 843),
 ('да', 820),
 ('к', 812),
 ('так', 809)]


**Выбор обработки пунктуации.**

Знаки препинания **отброшены**. Токенами считаются только слова найденные 
при помощи регулярного выражения `[а-яё]+(?:-[а-яё]+)*`.

Причины:
1. Пунктуация как отдельные токены увеличила бы размер словаря, но почти
все такие "слова" имели бы частоту 1–2 и не давали бы статистически 
осмысленных биграмм.
2. Запятые и точки в методе `generate` дают текст, забитый знаками, который трудно
читать. Для сравнения температур полезнее "чистый" лексический поток.
3. Для метода`logprob` на отложенной части пунктуация тоже не нужна: она не
различает "связный" и "перемешанный" текст, а только добавляет шум
одинакового порядка в оба варианта.